In [ ]:
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from pydantic import BaseModel, Field
import operator

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="zai-org/GLM-5.3-Flash",
    task="text-generation",
)

# You can use any model you want
model = ChatHuggingFace(llm=llm)


In [26]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(..., description="The detailed feedback about the essay.")
    score: int = Field(..., description="The score of the essay on a scale of 1 to 10.", ge=0, le=10)

In [27]:
structured_output = model.with_structured_output(EvaluationSchema, method="json_schema")

In [28]:
essay = """Air pollution is a major environmental issue that affects the health and well-being of humans, animals, and plants. It is caused by the release of harmful substances into the atmosphere, such as carbon monoxide, sulfur dioxide, nitrogen oxides, and particulate matter. These pollutants can come from various sources, including industrial activities, transportation, and agricultural practices.it can lead to respiratory problems, cardiovascular diseases, and even premature death. Additionally, air pollution contributes to climate change and environmental degradation. To mitigate air pollution, it is essential to implement stricter regulations on emissions, promote the use of clean energy sources, and raise public awareness about the importance of reducing pollution levels."""

In [29]:
essay

'Air pollution is a major environmental issue that affects the health and well-being of humans, animals, and plants. It is caused by the release of harmful substances into the atmosphere, such as carbon monoxide, sulfur dioxide, nitrogen oxides, and particulate matter. These pollutants can come from various sources, including industrial activities, transportation, and agricultural practices.it can lead to respiratory problems, cardiovascular diseases, and even premature death. Additionally, air pollution contributes to climate change and environmental degradation. To mitigate air pollution, it is essential to implement stricter regulations on emissions, promote the use of clean energy sources, and raise public awareness about the importance of reducing pollution levels.'

In [30]:
import time

prompt = f"Evaluate the following essay and provide detailed feedback along with a score out of 10. Essay: {essay}"

# Add a delay to avoid rate limiting
time.sleep(5)  # Wait 5 seconds before making the request

try:
    result = structured_output.invoke(prompt)
    print(result)
except Exception as e:
    print(f"Error: {e}")
    print("Please wait a moment and try again - you may have hit the API rate limit")

Error: (Request ID: Root=1-6a9482f4-32e8911c40de9b0455c000a7;750674f4-8c29-49a1-ad9e-2b40036464be)

429 Too Many Requests for url: https://router.huggingface.co/v1/chat/completions.
Rate limit exceeded
Please wait a moment and try again - you may have hit the API rate limit


In [31]:
class MyState(TypedDict):
    essay: str
    feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback:str
    individual_scores:Annotated[list[int],operator.add]
    avg_score:float

In [32]:
def evaluate_language(state: MyState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_output.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [33]:
def evaluate_analysis(state: MyState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_output.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [34]:
def evaluate_thought(state: MyState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_output.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [35]:
def final_evaluation(state: MyState):

    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}
    

In [36]:
graph = StateGraph(MyState)


graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
try:
    initial_state = MyState(
        essay=essay,
    )
    workflow.invoke(initial_state)
except Exception as e:
    print(f"Error: {e}")
    print("Please wait a moment and try again - you may have hit the API rate limit")                                                                                              

Error: Invalid json output: 
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
Please wait a moment and try again - you may have hit the API rate limit
